In [23]:
import pandas as pd
import numpy as np 
import sys
import os
from pathlib import Path 
from tqdm import tqdm 

In [24]:
path_to_results = Path("../../project_folder/results/mi_experiments/classifier/gaussians")

In [25]:
results_baseline_clf = {"mse_test": [],
                        "mse_val": [],
                        "num_dims": [], 
                        "hidden_dim": [],
                        "hidden_layers": [], 
                        "iteration": [],
                        "loc": []
                       }

for folder in tqdm(os.listdir(path_to_results)):
    folder_split = folder.split("_")
    try:
        loc, dimensions, hidden_dim, hidden_layers_no, iteration = (float(folder_split[1]), 
                                                                    int(folder_split[3]), 
                                                                    int(folder_split[6]), 
                                                                    int(folder_split[10]),
                                                                    int(folder_split[12]))
        metrics = pd.read_csv(path_to_results / folder / "metrics.csv")
        val_mse = metrics.mse_log_ratios_val[0].item()
        test_mse = metrics.mse_log_ratios_test[0].item()
        results_baseline_clf["mse_test"].append(test_mse)
        results_baseline_clf["mse_val"].append(val_mse)
        results_baseline_clf["num_dims"].append(dimensions)
        results_baseline_clf["hidden_dim"].append(hidden_dim)
        results_baseline_clf["hidden_layers"].append(hidden_layers_no)
        results_baseline_clf["iteration"].append(iteration)
        results_baseline_clf["loc"].append(loc)
    except:
        continue

100%|██████████| 376/376 [00:17<00:00, 21.69it/s]


In [26]:
results_baseline_clf_df = pd.DataFrame(results_baseline_clf)

In [27]:
# Average out the iteration axis 
df_mean = (
    results_baseline_clf_df
    .groupby(["hidden_dim", "hidden_layers", "num_dims", "loc"], as_index=False)
    .mean()   
)

In [28]:
best_configs = df_mean.loc[
    df_mean.groupby(["loc", "num_dims"])["mse_val"].idxmin()
]

In [29]:
best_configs

,hidden_dim,hidden_layers,num_dims,loc,mse_test,mse_val,iteration
45,128,0,2,1.0,0.000173,0.000173,1.0
47,128,0,5,1.0,0.001535,0.001520,1.0
6,64,0,10,1.0,0.007513,0.007370,1.0
51,128,0,20,1.0,0.071503,0.073151,1.0
53,128,0,30,1.0,1.009372,1.000282,1.0
1,64,0,2,2.0,0.000340,0.000341,1.0
4,64,0,5,2.0,0.012071,0.011852,1.0
7,64,0,10,2.0,16.515366,16.670297,1.0
42,64,3,20,2.0,145.286225,144.979234,1.0
124,256,3,30,2.0,544.635217,541.797891,1.0


In [30]:
best_configs_to_save = best_configs.copy()
best_configs_to_save["value"] = best_configs["mse_test"]
best_configs_to_save = best_configs_to_save.drop(columns=["mse_test", "mse_val", "iteration"])

In [31]:
best_configs_to_save

,hidden_dim,hidden_layers,num_dims,loc,value
45,128,0,2,1.0,0.000173
47,128,0,5,1.0,0.001535
6,64,0,10,1.0,0.007513
51,128,0,20,1.0,0.071503
53,128,0,30,1.0,1.009372
1,64,0,2,2.0,0.000340
4,64,0,5,2.0,0.012071
7,64,0,10,2.0,16.515366
42,64,3,20,2.0,145.286225
124,256,3,30,2.0,544.635217


In [32]:
best_configs_to_save.to_csv(path_to_results / "gaussian_results.csv")

In [20]:
results_baseline_clf_df_zeros = results_baseline_clf_df[results_baseline_clf_df.hidden_layers==2]

In [21]:
results_baseline_clf_df_zeros.groupby(["hidden_dim", "loc", "num_dims"]).mean()

mse_test       mse_val  hidden_layers  iteration
hidden_dim loc num_dims                                                      
64         1.0 2             0.008345      0.008324            2.0        1.0
               5             0.037978      0.038434            2.0        1.0
               10            0.557821      0.551966            2.0        1.0
               20          112.250223    108.787549            2.0        1.0
               30         4404.951128   4365.122183            2.0        1.0
           2.0 2             0.031545      0.032316            2.0        1.0
               5             8.452320      8.278096            2.0        1.0
               10          773.689057    792.035437            2.0        1.0
               20          209.083138    211.116277            2.0        1.0
               30          685.952516    682.781911            2.0        1.0
128        1.0 2             0.008369      0.008433            2.0        1.0
               5             0.056102      0.056916            2.0        1.0
               10            1.052960      1.057363            2.0        1.0
               20         1259.928727   1233.262877            2.0        1.0
               30        11668.826682  11572.479485            2.0        1.0
           2.0 2             0.070918      0.071143            2.0        1.0
               5            29.961724     29.492736            2.0        1.0
               10         3841.652644   3926.819548            2.0        1.0
               20          286.193011    287.891118            2.0        1.0
               30          648.532178    646.232056            2.0        1.0
256        1.0 2             0.007028      0.007006            2.0        1.0
               5             0.054340      0.055034            2.0        1.0
               10            1.357933      1.338429            2.0        1.0
               20         2845.156013   2775.523518            2.0        1.0
               30        16760.312799  16562.589487            2.0        1.0
           2.0 2             0.040158      0.040612            2.0        1.0
               5            86.197526     84.968857            2.0        1.0
               10         7904.677492   8077.717146            2.0        1.0
               20          216.671479    218.596555            2.0        1.0
               30          559.740495    557.570051            2.0        1.0

In [22]:
pred = np.load("/home/icb/alessandro.palma/environment/scRatio/project_folder/results/mi_experiments/classifier/gaussians/loc_1.0_dimensions_30_hidden_dim_64_hidden_layers_no_0_iteration_0/log_ratios_test.npy")
true = np.load("/home/icb/alessandro.palma/environment/scRatio/project_folder/results/mi_experiments/classifier/gaussians/loc_1.0_dimensions_30_hidden_dim_64_hidden_layers_no_0_iteration_0/true_log_ratios_test.npy")

In [14]:
np.mean((pred - true)**2)

np.float64(0.9968094676388617)

In [17]:
pred

array([ 11.452053 , -20.93195  ,  14.542465 , ...,  10.632149 ,
        12.090984 ,  -6.1843534], dtype=float32)

In [18]:
true

array([ 12.1886201 , -21.54404227,  15.47800852, ...,  11.75315937,
        12.49518044,  -6.45019733])

,mse_test,mse_val,num_dims,hidden_dim,hidden_layers,iteration,loc
0,1602.292499,1591.930133,30,64,0,2,2.0
1,0.001315,0.001297,5,256,0,1,1.0
2,0.003685,0.003641,2,128,1,2,1.0
3,0.047810,0.048116,2,256,2,0,2.0
4,0.067178,0.068327,5,128,2,2,1.0
...,...,...,...,...,...,...,...
355,11727.199084,11500.878183,20,256,3,1,1.0
356,274.146189,277.573564,20,128,1,0,2.0
357,8865.343395,9069.134671,10,256,2,0,2.0
358,4548.001130,4533.839733,30,64,2,1,1.0
